# 03 · Baselines E0 (validación 2024)
E0 no entrena nada: predice cada mes con reglas fijas sobre el historial. Son los cinco baselines de las directrices más `media_12_estacional`, añadido para corregir el efecto agosto. Es el suelo con el que se comparan los demás modelos.
Lógica en `src/features.py`, `src/metricas.py` y `src/baselines.py`.

**Protocolo:** solo se evalúan los 12 meses de **2024** (validación). **2025 (test) no se toca.** Para cada mes, la predicción usa únicamente meses anteriores (latencia 1: se conoce el mes previo).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

from baselines import BASELINES, evaluar_e0

PROCESSED = ROOT / "data" / "processed"
VALIDACION = pd.date_range("2024-01-01", "2024-12-01", freq="MS")
BASELINES

## Resultados por resolución
- **MAE / RMSE:** error medio (menos es mejor). **Devianza de Poisson:** pensada para conteos; penaliza mucho predecir 0 cuando hubo accidentes.
- **Capture@10 %:** proporción de los accidentes reales que caen en el 10 % de celdas con mayor predicción (más es mejor). Con 21 distritos el 10 % son 3 zonas (el 14 %), así que **no es comparable entre resoluciones**.

In [ ]:
paneles = {n: pd.read_parquet(PROCESSED / f"panel_{n}.parquet") for n in ["1km", "2km", "distritos"]}
resultados = {n: evaluar_e0(p, VALIDACION).set_index("baseline").round(3) for n, p in paneles.items()}
resultados["1km"]

In [ ]:
resultados["2km"]

In [ ]:
resultados["distritos"]

## Comparativa (MAE, 1 km)
`media_12_estacional` (media de 12 meses × índice estacional del mes, estimado solo con años pasados y sin 2020) es el baseline a batir. Detalle del efecto agosto en el notebook 04.

In [ ]:
resultados["1km"]["MAE"].sort_values().plot.barh(figsize=(6, 3), title="MAE en 2024 (1 km)")